<a href="https://colab.research.google.com/github/look4pritam/SurrogateModel/blob/master/Notebooks/SurrogateModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Surrogate Model

## Install Python packages.

In [1]:
!pip3 install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 39.9 MB/s eta 0:00:00


## Import Python packages.

In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

## Set random seed.

In [3]:
seed = 42
pl.seed_everything(seed, workers=True)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_float32_matmul_precision("high")

INFO:lightning_fabric.utilities.seed:Seed set to 42


## Set CPU or GPU device.

In [4]:
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
NUM_DEVICES = torch.cuda.device_count() if torch.cuda.is_available() else 1

## Define dataset.

In [5]:
class CSVDataset(Dataset):

    def __init__(self, file):

        data = pd.read_csv(file)
        data = data.sample(frac=1, random_state=seed).reset_index(drop=True)

        # INPUTS (x1 to x5)
        X = data[['x1', 'x2', 'x3', 'x4', 'x5']].values
        y = data[['y']].values

        # Normalize inputs
        self.x_mean = X.mean(axis=0)
        self.x_std = X.std(axis=0) + 1e-8
        X = (X - self.x_mean) / self.x_std

        self.x = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

## Define data module.

In [6]:
class CSVDataModule(pl.LightningDataModule):

    def __init__(self, file):
        super().__init__()
        self.file = file

    def setup(self, stage=None):

        dataset = CSVDataset(self.file)

        total = len(dataset)
        train_size = int(0.8 * total)
        val_size = int(0.1 * total)
        test_size = total - train_size - val_size

        self.train_dataset, self.val_dataset, self.test_dataset = random_split(
            dataset,
            [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(seed)
        )

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=32, shuffle=True, num_workers=4)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=32, shuffle=False, num_workers=4)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=32, shuffle=False, num_workers=4)

## Define a model.

In [7]:
class Net(pl.LightningModule):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):

        x, y = batch
        pred = self(x)
        loss = self.criterion(pred, y)

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):

        x, y = batch
        pred = self(x)
        loss = self.criterion(pred, y)

        self.log("val_loss", loss, prog_bar=True)

    def test_step(self, batch, batch_idx):

        x, y = batch
        pred = self(x)
        loss = self.criterion(pred, y)

        self.log("test_loss", loss)

    def configure_optimizers(self):

        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            patience=10
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler,
            "monitor": "val_loss"
        }

## Create the model.

In [8]:
model = Net()

## Create a data module.

In [9]:
data_module = CSVDataModule("dataset.csv")

## Create callback functions.

In [10]:
early_stop = EarlyStopping(monitor="val_loss", patience=30, mode="min")

checkpoint = ModelCheckpoint(
    dirpath="./models/",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="model"
)

## Create a model trainer.

In [11]:
trainer = pl.Trainer(
    max_epochs=200,
    accelerator=DEVICE,
    devices=NUM_DEVICES,
    precision="16-mixed" if DEVICE == "gpu" else 32,
    callbacks=[early_stop, checkpoint],
    deterministic=True,
    default_root_dir="./models/logs",
    log_every_n_steps=10
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Train the model.

In [12]:
trainer.fit(model, data_module)

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/models exists and is not empty.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net       │ Sequential │  6.7 K │ train │     0 │
│ 1 │ criterion │ MSELoss    │      0 │ train │     0 │
└───┴───────────┴────────────┴────────┴───────┴───────┘

Trainable params: 6.7 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.7 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will 
create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller 
than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 
running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will 
create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller 
than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 
running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=200` reached.


## Test the model.

In [13]:
trainer.test(ckpt_path="best", datamodule=data_module)

INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/models/model.ckpt
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/models/model.ckpt
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: Thi

Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   0.058374837040901184    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.058374837040901184}]

## Save the model.

In [14]:
best_model = Net.load_from_checkpoint(checkpoint.best_model_path)
best_model.eval()

scripted_model = torch.jit.script(best_model.net)
torch.jit.save(scripted_model, "./models/model.pt")